## 07 · 用 LangChain 表达同一条 RAG 链

第 06 课用普通 Python 写完了 RAG。这里不增加新能力，只用 LangChain Core 的数据流写法表达同一件事，方便看清框架到底替你连接了什么。

In [1]:
import os
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

from dotenv import load_dotenv

load_dotenv("../.env")
api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
if api_key:
    os.environ["OPENAI_API_KEY"] = api_key
if os.getenv("LLM_BASE_URL"):
    os.environ["OPENAI_BASE_URL"] = os.getenv("LLM_BASE_URL")

question = "SKU-JK902 的防水性能是什么？"
retrieved = [
    Document(
        page_content="冲锋衣 JK902 采用三层防水面料，静水压达到 20,000 mm。",
        metadata={"source": "产品/冲锋衣-JK902/产品规格.md"},
    )
]

def format_docs(docs):
    return "\n\n".join(
        f"[来源：{doc.metadata['source']}]\n{doc.page_content}" for doc in docs
    )

retriever = RunnableLambda(lambda _: retrieved)
prompt = ChatPromptTemplate.from_template(
    "请只根据资料回答问题；没有答案时请说不知道。\n\n资料：{context}\n\n问题：{question}"
)
rag_prompt = {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt

if api_key:
    if not os.getenv("LLM_MODEL"):
        raise RuntimeError("请在 .env 中配置 LLM_MODEL。")
    from langchain_openai import ChatOpenAI
    chain = rag_prompt | ChatOpenAI(model=os.environ["LLM_MODEL"], temperature=0) | StrOutputParser()
    print(chain.invoke(question))
else:
    print(rag_prompt.invoke(question).to_string())


Human: 请只根据资料回答问题；没有答案时请说不知道。

资料：[来源：产品/冲锋衣-JK902/产品规格.md]
冲锋衣 JK902 采用三层防水面料，静水压达到 20,000 mm。

问题：SKU-JK902 的防水性能是什么？


上面的 `retriever` 暂时返回一段已经准备好的结果，目的是只看懂 LCEL（LangChain Expression Language，LangChain 的连接语法）如何把“检索结果 → 提示词”串起来。真实向量检索仍然是第 06 课的核心。